<a href="https://colab.research.google.com/github/perfectmakuwerere/efficient-llm-lab/blob/main/SmolLM2_(function_calling_finetuning).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
HF_USERNAME  = "nakue"
BASE_MODEL   = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
DATASET      = "hiyouga/glaive-function-calling-v2-sharegpt"
MAX_SEQ_LEN  = 2048
MAX_STEPS    = 300
LORA_RANK    = 16

FT_BF16_REPO = f"{HF_USERNAME}/SmolLM2-1.7B-FT-BF16"
FT_W8A8_REPO = f"{HF_USERNAME}/SmolLM2-1.7B-FT-W8A8"

In [9]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 3.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 145.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 146.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB

In [2]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import torch

print("Loading base model...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=torch.bfloat16,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# Apply ChatML template — SmolLM2 uses this format
tokenizer = get_chat_template(tokenizer, chat_template="chatml")

print(f"Model ready. Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading base model...
==((====))==  Unsloth 2026.6.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

HuggingFaceTB/SmolLM2-1.7B-Instruct does not have a padding token! Will use pad_token = <|endoftext|>.


Unsloth 2026.6.8 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.
Unsloth: Will map <|im_end|> to EOS = <|im_end|>.


Model ready. Trainable params: 18,087,936


In [3]:
from datasets import load_dataset

print("\nLoading dataset...")

dataset = load_dataset(DATASET, split="train")
print(f"Loaded {len(dataset):,} examples")

def format_conversations(examples):
    """
    glaive-sharegpt stores conversations as:
      [{"from": "system", "value": "..."},
       {"from": "human", "value": "..."},
       {"from": "gpt",   "value": "..."}]

    ChatML expects roles: system / user / assistant
    So we remap: human → user, gpt → assistant
    """
    texts = []
    for convo in examples["conversations"]:
        mapped = []
        for turn in convo:
            role = turn["from"]
            if role == "human":
                role = "user"
            elif role == "gpt":
                role = "assistant"
            mapped.append({"role": role, "content": turn["value"]})

        try:
            text = tokenizer.apply_chat_template(
                mapped,
                tokenize=False,
                add_generation_prompt=False,
            )
            texts.append(text)
        except Exception:
            texts.append("")  # skip malformed examples

    return {"text": texts}

dataset = dataset.map(format_conversations, batched=True)
dataset = dataset.filter(lambda x: len(x["text"]) > 10)
print(f"Dataset after formatting: {len(dataset):,} examples")



Loading dataset...


README.md:   0%|          | 0.00/433 [00:00<?, ?B/s]

glaive_toolcall.json:   0%|          | 0.00/251M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100563 [00:00<?, ? examples/s]

Loaded 100,563 examples


Map:   0%|          | 0/100563 [00:00<?, ? examples/s]

Filter:   0%|          | 0/100563 [00:00<?, ? examples/s]

Dataset after formatting: 100,563 examples


In [4]:
from trl import SFTTrainer, SFTConfig

print("\nStarting fine-tuning...")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,   # effective batch = 16
        warmup_steps=10,
        max_steps=MAX_STEPS,
        learning_rate=2e-4,
        bf16=True,
        logging_steps=10,
        output_dir="checkpoints",
        optim="adamw_8bit",
        max_seq_length=MAX_SEQ_LEN,
        packing=True,
        save_strategy="no",
        report_to="none",
    ),
)

stats = trainer.train()
print(f"\nDone. Final loss: {stats.metrics['train_loss']:.4f}")


Starting fine-tuning...
Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/100563 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=16):   0%|          | 0/100563 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 20,892 | Num Epochs = 1 | Total steps = 300
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 18,087,936 of 1,729,464,320 (1.05% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,1.037563
20,0.866043
30,0.789957
40,0.752154
50,0.695315
60,0.707346
70,0.668430
80,0.666453
90,0.636607
100,0.631409



Done. Final loss: 0.6469


In [5]:
model.push_to_hub("nakue/SmolLM2-1.7B-LoRA-adapter")
tokenizer.push_to_hub("nakue/SmolLM2-1.7B-LoRA-adapter")

README.md:   0%|          | 0.00/557 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   2%|1         | 1.12MB / 72.4MB            

Saved model to https://huggingface.co/nakue/SmolLM2-1.7B-LoRA-adapter


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpa6zhekx6/tokenizer_config.json.


In [6]:
print("\nMerging LoRA adapters and saving (Unsloth dequant path)...")

model.save_pretrained_merged(
    "SmolLM2-1.7B-FT-BF16",
    tokenizer,
    save_method="merged_16bit",
)

print(f"Pushing to HF: {FT_BF16_REPO}")
model.push_to_hub_merged(
    FT_BF16_REPO,
    tokenizer,
    save_method="merged_16bit",
)
print(f"Done → https://huggingface.co/{FT_BF16_REPO}")


Merging LoRA adapters and saving (Unsloth dequant path)...


Unsloth: Restored added_tokens_decoder metadata in SmolLM2-1.7B-FT-BF16/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `SmolLM2-1.7B-FT-BF16`: 100%|██████████| 1/1 [00:05<00:00,  5.51s/it]


Successfully copied all 1 files from cache to `SmolLM2-1.7B-FT-BF16`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...FT-BF16/model.safetensors:   0%|          | 24.6kB / 3.42GB            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:05<00:00, 65.05s/it]


Unsloth: Merge process complete. Saved to `/content/nakue/SmolLM2-1.7B-FT-BF16`
Done → https://huggingface.co/nakue/SmolLM2-1.7B-FT-BF16
